In [ ]:
import pandas as pd

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_risk_good_train.csv"
df = pd.read_csv(fp)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clfrn = RandomForestClassifier(n_estimators=250, random_state=42)

In [ ]:
preds = df.columns.tolist()
preds.remove("LoanStatus")

In [ ]:
X_df = df[preds]
Y_rn = df.LoanStatus
#clfrn.fit(X_df, Y_rn)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
calibrated_clf = CalibratedClassifierCV(clfrn, method="sigmoid", cv=5)
calibrated_clf.fit(X_df, Y_rn)

In [ ]:
pp_rn = Y_rn.value_counts()[1]/Y_rn.value_counts()[0]

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
dfv = pd.read_csv(fp)
X_dfv = dfv[preds]
Y_v = dfv["LoanStatus"]

In [ ]:
val_res = {"prob_chgoff": calibrated_clf.predict_proba(X_dfv)[:, 1],
                "prob_PIF": calibrated_clf.predict_proba(X_dfv)[:, 0],
               "LoanStatus": Y_v}
df_val_res = pd.DataFrame.from_dict(val_res, orient="columns")

In [ ]:
df_val_res.LoanStatus.value_counts()

In [ ]:
from sklearn.metrics import precision_recall_curve,auc

In [ ]:
y_scores = calibrated_clf.predict_proba(X_dfv)[:, 1]

In [ ]:
# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(Y_v, y_scores)

In [ ]:
TN = 100
pTN = precision[:TN]
rTN = recall[:TN]
tTN = thresholds[:TN]
df_th_res = pd.DataFrame.from_dict({"thresh": tTN, "precision": pTN, "recall": rTN}, orient="columns")

In [ ]:
df_th_res.tail(20)

In [ ]:
THSEL = 0.0081

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(thresholds, precision[:-1], 'b--', label='Precision', marker="x")
plt.plot(thresholds, recall[:-1], 'r--', label='Recall', marker="o")
plt.xlabel('Threshold')
plt.legend(loc='lower left')
plt.ylim([0,1])
plt.grid(True)

In [ ]:
from sklearn.calibration import CalibrationDisplay
disp = CalibrationDisplay.from_predictions(Y_v, y_scores)
plt.grid(True)
plt.show()

In [ ]:
df_val_res["prediction"] = df_val_res["prob_chgoff"].apply(lambda x: 0 if x < THSEL else 1)

In [ ]:
df_val_res.prediction.value_counts()

In [ ]:
df_val_res.LoanStatus.value_counts()

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test = pd.read_csv(fp)

In [ ]:
Xt = df_test[preds]
Yt = df_test.LoanStatus

In [ ]:
test_res = {"prob_chgoff": calibrated_clf.predict_proba(Xt)[:, 1],
                "prob_PIF": calibrated_clf.predict_proba(Xt)[:, 0],
               "LoanStatus": Yt}
df_test_res = pd.DataFrame.from_dict(test_res, orient="columns")

In [ ]:
df_test_res["prediction"] = df_test_res["prob_chgoff"].apply(lambda x: 0 if x < THSEL else 1)

In [ ]:
test_pred_counts = df_test_res.prediction.value_counts()
test_pred_counts[1]/ (test_pred_counts[1] + test_pred_counts[0])

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(df_test_res.LoanStatus, df_test_res.prediction))